In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

#spilloverを考慮したSCM感度分析
"""
この感度分析の目的は、「ドナー群（比較対象）に介入の波及効果があるときに、SCMの推定精度がどれほど悪化するか」を確認すること。
SCMの計算結果は「処置群の実績値」と「合成コントロール（反実仮想）の予測値」の差分（Gap）の平均値。
そのため商圏が重複する対照群への流出によって処置群の数値が減っているかどうかに関わらず、
「ドナー側の来訪者が増えてしまうこと自体が分析の敵（処置効果の過小評価につながってしまう）」であるため、
まずは介入によるドナー側の増加（汚染）のみをシンプルにシミュレーションする
"""
input_path = Path("/content/drive/MyDrive/因果推論/h3_mesh_panel.csv")
output_dir = input_path.parent

intervention_date = pd.Timestamp("2025-01-01")
#介入の影響が処置を受けていない周辺地域（ドナーユニット）にどれくらいの強さで波及すると仮定するかを示すパラメータ
spillover_strengths = [0.00, 0.25, 0.50, 1.00]
#処置エリアとの商圏重複率（overlap_score）がこの値以上であるドナーユニットを、分析から除外するためのしきい値
#スピルオーバーの影響を受けている可能性が高いドナー（対照群）を、分析から除外するためのしきい値
exclusion_thresholds = [1.01, 0.70, 0.60, 0.50, 0.30]

#このCSVには実測の距離・隣接関係・商圏重複率がないため、教材用にdonor_groupとJaccard係数を設定する。駅名や実地域は表さない。
teaching_groups = {
    "donor_group_1": {"h3_ids": [f"h3_mesh_{i:02d}" for i in range(7, 13)], "overlap_score": 0.75},
    "donor_group_2": {"h3_ids": [f"h3_mesh_{i:02d}" for i in range(13, 19)], "overlap_score": 0.45},
    "donor_group_3": {"h3_ids": [f"h3_mesh_{i:02d}" for i in range(19, 25)], "overlap_score": 0.25},
    "donor_group_4": {"h3_ids": [f"h3_mesh_{i:02d}" for i in range(25, 31)], "overlap_score": 0.20},
    "donor_group_5": {"h3_ids": [f"h3_mesh_{i:02d}" for i in range(31, 37)], "overlap_score": 0.65},
}


def rmspe(values):
    values = np.asarray(values, dtype=float)
    return float(np.sqrt(np.mean(values ** 2)))

#spillover_strengths と exclusion_thresholds のすべての組み合わせ、つまり各シナリオに対して
#合成コントロール法の計算を行い、その結果を返すfit_scm()関数を定義
def fit_scm(panel, strength, threshold, true_effect):
    work = panel.copy()
    #work["assumed_spillover"] :仮定された波及効果によるドナーユニットの来訪者数の「かさ上げ分」を人工的に作り出した値。
    work["assumed_spillover"] = np.where((work["treated"] == 0) & (work["date"] >= intervention_date),
        #商圏重複率 * 未処置の領域に波及して影響を仮定した効果 * 真の処置効果
        work["overlap_score"] * strength * true_effect,0.0)
    #work["visitors_scenario"] はドナー群に波及効果の増加分を踏まえた観測者数
    work["visitors_scenario"] = work["visitors"] + work["assumed_spillover"]
    #対照群のうち、商圏重複率が特定のしきい値以上であるドナーグループを特定し、それらを分析から除外するリストを作成する
    excluded_groups = sorted(
        work.loc[(work["treated"] == 0) & (work["overlap_score"] >= threshold),"donor_group"
        ].dropna().unique())
    #商圏重複率のしきい値を変えて、商圏重複率が大きいエリアを段階的に除外してみる。
    donor_part = work.loc[(work["treated"] == 0) & (work["overlap_score"] < threshold)]
    donor_ids = sorted(donor_part["h3_id"].unique())
    if not donor_ids:
        raise ValueError(f"threshold={threshold}ではドナーが0件です。")
    #処置系列を作成
    actual = work.loc[work["treated"] == 1].groupby("date")["visitors_scenario"].mean().sort_index()
    #商圏重複率の閾値を超えたメッシュ除外後のドナー系列を作成
    donor_wide = (
        donor_part.pivot(index="date", columns="h3_id", values="visitors_scenario")
        .sort_index().loc[:, donor_ids]
    )
    dates = actual.index.intersection(donor_wide.index)
    actual = actual.loc[dates]
    donor_wide = donor_wide.loc[dates]
    pre_mask = dates < intervention_date
    #pre_mask のブール値を反転させたもの（論理否定)
    post_mask = ~pre_mask
    y_pre = actual.loc[pre_mask].to_numpy(float)
    x_pre = donor_wide.loc[pre_mask].to_numpy(float)

    def objective(weights):
        residual = y_pre - x_pre @ weights
        return np.mean(residual ** 2) / np.var(y_pre)

    donor_count = len(donor_ids)
    result = minimize(
        objective,
        np.full(donor_count, 1.0 / donor_count),
        method="SLSQP",
        bounds=[(0.0, 1.0)] * donor_count,
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        options={"ftol": 1e-12, "maxiter": 5000, "disp": False}
    )
    if not result.success:
        raise RuntimeError(f"SCM最適化失敗: {result.message}")

    weights = result.x
    synthetic = donor_wide.to_numpy(float) @ weights
    series = pd.DataFrame({
        "date": dates,
        "actual_treated_area": actual.to_numpy(float),
        "synthetic_treated_area": synthetic
    })
    series["gap"] = series["actual_treated_area"] - series["synthetic_treated_area"]
    series["period"] = np.where(series["date"] < intervention_date, "pre", "post")
    pre_gap = series.loc[series["period"] == "pre", "gap"]
    post_gap = series.loc[series["period"] == "post", "gap"]
    pre_rmspe = rmspe(pre_gap)
    post_rmspe = rmspe(post_gap)
    scenario_id = f"strength_{strength:.2f}__threshold_{threshold:.2f}"
    series["scenario_id"] = scenario_id
    display(series)

    summary = {
        "scenario_id": scenario_id,
        "spillover_strength": strength,
        "exclusion_threshold": threshold,
        "excluded_groups": "|".join(excluded_groups) if excluded_groups else "none",
        "donor_mesh_count": donor_count,
        "pre_rmspe": pre_rmspe,
        "post_rmspe": post_rmspe,
        "post_pre_rmspe_ratio": post_rmspe / pre_rmspe,
        "mean_post_gap": float(post_gap.mean()),
        "true_direct_effect": true_effect,
        "estimation_error": float(post_gap.mean() - true_effect),
    }
    weights_df = pd.DataFrame({
        "scenario_id": scenario_id,
        "donor_h3_id": donor_ids,
        "weight": weights
    })
    display(weights_df)
    return summary, series, weights_df

